
# Regularized Regression

## How do Models Learn? (The Engine)

Before understanding Regularization, we must peek under the hood of Linear Regression. How does the computer find the "best" line?

### 1. The Cost Function ($J$)

The computer needs a score to know how bad its predictions are. We call this the ****Cost Function**** (or Loss Function). For regression, we use ****Mean Squared Error (MSE)****:

$$J(w) = \frac{1}{2m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

*Note*: The $\frac{1}{2}$ factor is a convenience which cancels with the exponent when computing derivatives, making the math cleaner. The minimum stays in the same place.

Think of this as a "Bowl." The lowest point of the bowl represents the smallest error.

### 2. Gradient Descent

To find the bottom of the bowl, algorithms use ****Gradient Descent****.

1.  Start at a random point (random weights).
2.  Calculate the slope (gradient).
3.  Take a step downhill.
4.  Repeat until you reach the bottom.

## The Problem: Overfitting

Sometimes, the model tries too hard. It finds a complex "bottom of the bowl" that fits the training data perfectly (including the noise) but fails on new data. This is ****Overfitting****.

## The Solution: Regularization

Regularization acts as a "penalty" or a "tax" on complexity. We modify the Cost Function:

$$\text{Total Cost} = \text{Error (MSE)} + \text{Penalty}$$

The model now has to balance two goals:

1.  Fit the data well (minimize Error).
2.  Keep coefficients simple (minimize Penalty).

### Types of Regularization

| Type      | Name           | Penalty Formula               | Effect                             | Best For                                                           |
|--------- |-------------- |----------------------------- |---------------------------------- |------------------------------------------------------------------ |
| **L1**    | **Lasso**      | $\lambda \sum\lvert w \rvert$ | Pushes coefficients to **Zero**.   | **Feature Selection**: When you suspect many features are useless. |
| **L2**    | **Ridge**      | $\lambda \sum w^2$            | Pushes coefficients **Near Zero**. | **Multicollinearity**: When features are highly correlated.        |
| **L1+L2** | **ElasticNet** | Combination                   | Hybrid of above.                   | When you have many features and correlations.                      |

**Note**: $\lambda$ (Lambda) is the "Tax Rate". High $\lambda$ = High Penalty = Simpler Model. In scikit-learn, this parameter is called `alpha`.

## Practical Demonstration: The "Sparse" Signal

We will demonstrate the superpower of **Lasso**: automatically removing useless features. We generate a dataset with **50 features**, but only **5** of them actually predict the target. The rest are pure noise.

### Generate Noisy Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

# Generate 50 features, but only 5 are informative
X, y, true_coef = make_regression(n_samples=100, n_features=50, n_informative=5, 
                                  noise=10, coef=True, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total Features: {X.shape[1]}")
print(f"Useful Features: 5")

### Linear Regression (The Control)

Standard regression tries to use **all** 50 features, fitting noise.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"Linear Train R²: {lr.score(X_train, y_train):.2f}")
print(f"Linear Test R²:  {lr.score(X_test, y_test):.2f}")

**Expected**: High Train score, lower Test score. Classic Overfitting.

### Lasso vs. Ridge

Let's see how they handle the noise.

In [ ]:
from sklearn.linear_model import Lasso, Ridge

# Lasso (L1) - High alpha for demonstration
lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)

# Ridge (L2)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

print(f"Lasso Test R²: {lasso.score(X_test, y_test):.2f}")
print(f"Ridge Test R²: {ridge.score(X_test, y_test):.2f}")

### ElasticNet: The Hybrid

ElasticNet combines both penalties. The `l1_ratio` parameter controls the mix:

-   `l1_ratio=1` → Pure Lasso
-   `l1_ratio=0` → Pure Ridge
-   `l1_ratio=0.5` → Equal mix

In [ ]:
from sklearn.linear_model import ElasticNet

# ElasticNet with 50% L1, 50% L2
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5)
elastic.fit(X_train, y_train)

print(f"ElasticNet Test R²: {elastic.score(X_test, y_test):.2f}")
print(f"ElasticNet features at exactly 0: {np.sum(elastic.coef_ == 0)}")

**Expected**: ElasticNet zeros out some features (like Lasso) while shrinking others smoothly (like Ridge).

### Visualizing Coefficients

This is the most important visualization in this module.

-   **Lasso** should have many coefficients exactly at 0.
-   **Ridge** should have many small coefficients, but none at 0.
-   **ElasticNet** should be in between: some zeros, some small values.

In [ ]:
plt.figure(figsize=(15, 4))

# Plot Lasso
plt.subplot(1, 3, 1)
plt.plot(lasso.coef_, marker='o', linestyle='none', color='crimson')
plt.title("Lasso Coefficients (L1)")
plt.xlabel("Feature Index")
plt.ylabel("Coefficient Value")
plt.axhline(0, color='black', linestyle='--')

# Plot Ridge
plt.subplot(1, 3, 2)
plt.plot(ridge.coef_, marker='o', linestyle='none', color='navy')
plt.title("Ridge Coefficients (L2)")
plt.xlabel("Feature Index")
plt.axhline(0, color='black', linestyle='--')

# Plot ElasticNet
plt.subplot(1, 3, 3)
plt.plot(elastic.coef_, marker='o', linestyle='none', color='green')
plt.title("ElasticNet Coefficients (L1+L2)")
plt.xlabel("Feature Index")
plt.axhline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()

# Count zero coefficients
print(f"Lasso features set to exactly 0: {np.sum(lasso.coef_ == 0)}")
print(f"Ridge features set to exactly 0: {np.sum(ridge.coef_ == 0)}")
print(f"ElasticNet features set to exactly 0: {np.sum(elastic.coef_ == 0)}")

## Exercises

Use **Lasso** to perform "Feature Selection" on the real California Housing dataset.

### Load Data

Load the data and add some random noise features to trick the model.

### Feature Scaling

Regularization penalizes coefficient magnitude. If features have different scales (e.g., income in thousands vs. age in years), the penalty unfairly targets high-scale features. Standardizing ensures all features compete equally:

### Tuning Alpha with GridSearch

We don't guess the penalty strength ($\lambda$ or `alpha`). We search for it. **Note**: `GridSearchCV` tries every combination of parameters to find the best one.

### Which features survived?

Retrieve the best model and inspect which coefficients survived. Did Lasso kill the "Noise" columns? **Expected Result**: The `Noise_X` features should have coefficients very close to or exactly Zero.

## Summary

1.  **Overfitting** happens when models memorize noise.
2.  **Regularization** prevents this by penalizing large coefficients.
3.  **Lasso (L1)** zeros out useless features (Feature Selection).
4.  **Ridge (L2)** shrinks all features together (good for correlation).
5.  **ElasticNet** combines both, controlled by `l1_ratio`.